In [1]:
import os, numpy as np, pandas as pd, joblib
from collections import defaultdict
from scipy.stats import hypergeom, fisher_exact
from statsmodels.stats.multitest import multipletests

OLD_BASE = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
# NOTE: new-427 genes live dir is being overwritten by the cap-400 rerun, so point at the cap750 BACKUP:
NEW_BASE = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/_backup_rosmap427_cap750/Multirun_cell_on_cell_genes_rosmap427"

canonical = ["Ast", "Mic", "Inh", "Oli", "Opc", "Ex"]
OLD_CT = {"Ast":"Ast","Mic":"Mic","Inh":"In", "Oli":"Oli","Opc":"Opc","Ex":"Ex"}   # old ROSMAP uses "In"
NEW_CT = {"Ast":"Ast","Mic":"Mic","Inh":"Inh","Oli":"Oli","Opc":"Opc","Ex":"Ex"}  # 427 uses "Inh"

def load_ml(ml_base, ct_folder):                      # predictor = importance>0 in >=2/5 splits
    counts = defaultdict(int); pool = set(); ns = 0
    for s in range(1, 6):
        p = os.path.join(ml_base, ct_folder, f"split_{s}", "maximal_classifier.joblib")
        if not os.path.exists(p): continue
        ns += 1; m = joblib.load(p)
        fn = [str(g) for g in m.feature_names_in_]; fi = np.asarray(m.feature_importances_).ravel()
        pool.update(fn)
        for g, imp in zip(fn, fi):
            if imp > 0: counts[g] += 1
    return {g for g,c in counts.items() if c >= 2}, pool, ns

old = {ct: load_ml(OLD_BASE, OLD_CT[ct]) for ct in canonical}
new = {ct: load_ml(NEW_BASE, NEW_CT[ct]) for ct in canonical}

rows, overlaps = [], {}
for ct in canonical:
    A, Apool, na = old[ct]; B, Bpool, nb = new[ct]
    N = len(Apool | Bpool); K, n = len(A), len(B); k = len(A & B)
    overlaps[ct] = sorted(A & B)
    if N and K and n:
        p_hyper = float(hypergeom.sf(k-1, N, K, n)); expected = K*n/N
        fold = k/expected if expected else np.nan
        _, p_fisher = fisher_exact([[k, K-k],[n-k, N-K-n+k]], alternative="greater")
    else:
        p_hyper = expected = fold = p_fisher = np.nan
    rows.append(dict(cell_type=ct, old_pred=K, new_pred=n, overlap=k,
                     expected=round(expected,2), fold=round(fold,2),
                     p_hyper=p_hyper, p_fisher=p_fisher))

res = pd.DataFrame(rows)
mask = res["p_hyper"].notna()
res.loc[mask,"p_hyper_FDR"] = multipletests(res.loc[mask,"p_hyper"], method="fdr_bh")[1]
print(res.to_string(index=False))
for ct in canonical:
    ov = overlaps[ct]; print(f"{ct}: {len(ov)} -> {ov[:40]}{' ...' if len(ov)>40 else ''}")


cell_type  old_pred  new_pred  overlap  expected  fold      p_hyper     p_fisher  p_hyper_FDR
      Ast       175       349       40     11.33  3.53 5.970584e-13 5.970584e-13 1.194117e-12
      Mic       466       255       85     24.76  3.43 4.407638e-27 4.407638e-27 2.644583e-26
      Inh       108        10        2      0.11 17.96 5.213152e-03 5.213152e-03 5.213152e-03
      Oli       621       179       62     20.52  3.02 7.149911e-17 7.149911e-17 2.144973e-16
      Opc       835       147       45     16.23  2.77 6.698831e-11 6.698831e-11 1.004825e-10
       Ex       162        11        5      0.15 32.89 2.047753e-07 2.047753e-07 2.457303e-07
Ast: 40 -> ['ADAM23', 'AKT3', 'COL5A3', 'CTNND2', 'DNAH7', 'ENHO', 'FBXL7', 'FTH1', 'FTL', 'GRM3', 'HIF3A', 'KANSL1', 'KCNIP4', 'LINGO1', 'MERTK', 'MRAS', 'MRPS6', 'MT1E', 'MT1G', 'MT1M', 'MT3', 'NRXN1', 'PALLD', 'PCDH9', 'PIK3C2A', 'PLCB1', 'PLCE1', 'RERG', 'RGS6', 'SLC14A1', 'SLC25A37', 'SON', 'STON2', 'SYTL4', 'TENM2', 'TNIK', 'TRPM3', '